In [24]:
import pandas as pd 
import numpy as np

In [25]:
df = pd.read_csv('results.csv')
df.head()
df['date'] = pd.to_datetime(df['date'])

In [26]:
conditions = [
    df['home_score'] > df['away_score'],
    df['home_score'] < df['away_score'],
    df['home_score'] == df['away_score']
]

results = [
    'home_win',
    'away_win',
    'tie'
]

df['Target'] = np.select(conditions, results, default='Error')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,Target
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,tie
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,tie
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win


In [27]:
def momentum (game_date, team_name):
   df_new = df[((game_date > df['date']) & ((game_date - pd.DateOffset(years=8)) < df['date'])) & ((team_name == df['home_team']) | (team_name == df['away_team']))]
   h_s = df_new[ df_new['home_team'] == team_name ]['home_score'].sum()
   a_s = df_new[ df_new['away_team'] == team_name]['away_score'].sum()
   goals_scored = h_s + a_s

   h_c = df_new[ df_new['home_team'] == team_name]['away_score'].sum()
   a_c = df_new[ df_new['away_team'] == team_name]['home_score'].sum()
   goals_conceded = h_c + a_c

   return goals_scored, goals_conceded

In [28]:
def head_to_head (game_date, team_a, team_b):
    df_h2h = df[(game_date > df['date']) & (
                # team_a at home, team_b away
                ((team_a == df['home_team']) & (team_b == df['away_team'])) 
                | # OR
                # team_b at home, team_a away
                ((team_a == df['away_team']) & (team_b == df['home_team']))
                )]
    
    a_wins = (
        # a - home
        ((df_h2h['home_team'] == team_a) & (df_h2h['Target'] == 'home_win'))
        | # OR
        # a - away
        ((df_h2h['away_team'] == team_a) & (df_h2h['Target'] == 'away_win'))
    ).sum()

    b_wins = (
        # b - home
        ((df_h2h['home_team'] == team_b) & (df_h2h['Target'] == 'home_win'))
        | # OR
        # b - away
        ((df_h2h['away_team'] == team_b) & (df_h2h['Target'] == 'away_win'))
    ).sum()

    ties = (df_h2h['Target'] == 'ties').sum()
 
    return a_wins, b_wins, ties

In [29]:
df[['h2h_home_wins', 'h2h_away_wins', 'h2h_ties']] = df.apply(lambda row: head_to_head(row['date'], row['home_team'], row['away_team']), axis=1, result_type='expand')

KeyboardInterrupt: 

In [ ]:
df[['home_scored', 'home_conceded']] = df.apply(lambda row: momentum(row['date'], row['home_team']), axis=1, result_type='expand')
df[['away_scored', 'away_conceded']] = df.apply(lambda row: momentum(row['date'], row['away_team']), axis=1, result_type='expand')

In [ ]:
df.head()